In [1]:
import json
from ultralytics import YOLO
from functions.benchmark import benchmark
from functions.export_to_tensorrt import export_to_tensorrt

validation_datasets = [
    # "coco.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_low/data_blurry_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_medium/data_blurry_medium.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_contrast_low/data_contrast_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_jpeg_heavy/data_jpeg_heavy.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_low/data_noisy_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_medium/data_noisy_medium.yaml"
    "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_mixed_degrad_50pct/data_coco_val_mixed_degrad_50pct.yaml",
]

benchmark_results = []

model = YOLO("yolo12x", task="detect")  # Load a model

exported_model_path = export_to_tensorrt(
    model
)

for validation_dataset in validation_datasets:
    benchmark_result = benchmark(
        model=YOLO(exported_model_path, task="detect"),
        kwargs={
            "data": validation_dataset,
        })
    
    results_data = {
        "mAP50-95": benchmark_result.results_dict.get('metrics/mAP50-95(B)', None),
        "mAP50": benchmark_result.results_dict.get('metrics/mAP50(B)', None),
        "latency_ms": benchmark_result.speed.get('inference', None),
        "fps": None,
    }
    if results_data["latency_ms"] is not None:
        results_data["fps"] = 1000.0 / results_data["latency_ms"]
    
    benchmark_results.append({
        "model": "yolo12x",
        "precision": "FP32",
        "dataset": validation_dataset,
        "benchmark_result": results_data,
    })
    
with open("benchmark_results_fp32.json", "w") as f:
    json.dump(benchmark_results, f, indent=4)

100%|██████████| 114M/114M [00:15<00:00, 7.62MB/s] 


Ultralytics 8.3.109 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2070, 7789MiB)
YOLOv12x summary (fused): 283 layers, 59,135,744 parameters, 0 gradients, 199.0 GFLOPs

PyTorch: starting from 'yolo12x.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (113.8 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.50...
ONNX: export success ✅ 4.8s, saved as 'yolo12x.onnx' (226.0 MB)

TensorRT: starting export with TensorRT 10.9.0.34...
[05/01/2025-17:12:17] [TRT] [I] [MemUsageChange] Init CUDA: CPU -2, GPU +0, now: CPU 2876, GPU 802 (MiB)
[05/01/2025-17:12:18] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU +123, GPU +192, now: CPU 2797, GPU 994 (MiB)
[05/01/2025-17:12:18] [TRT] [I] ----------------------------------------------------------------
[05/01/2025-17:12:18] [TRT] [I] Input filename:   yolo12x.onnx
[05/01/2025-17:12:18] [TRT] [I] ONNX IR version:  0.0.9
[05/01/2025-17:12:18] [TRT] [I] Opset v

val: Scanning /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_mixed_degrad_50pct/labels/val2017... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:01<00:00, 2509.95it/s]


val: New cache created: /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_mixed_degrad_50pct/labels/val2017.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5000/5000 [04:59<00:00, 16.69it/s]


                   all       5000      36335      0.715      0.606      0.657      0.498
Speed: 0.3ms preprocess, 56.7ms inference, 0.0ms loss, 1.0ms postprocess per image
Benchmark completed.
Benchmark results: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77,
       78, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x79269b4cc710>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008

In [ ]:
import json
from ultralytics import YOLO
from functions.benchmark import benchmark
from functions.export_to_tensorrt import export_to_tensorrt

calibration_sets = [
    "/home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_clean/data.yaml",
    "/home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_mixed/data.yaml",
]

validation_datasets = [
    # "coco.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_low/data_blurry_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_medium/data_blurry_medium.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_contrast_low/data_contrast_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_jpeg_heavy/data_jpeg_heavy.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_low/data_noisy_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_medium/data_noisy_medium.yaml"
    "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_mixed_degrad_50pct/data_coco_val_mixed_degrad_50pct.yaml",
]

benchmark_results = []

model = YOLO("yolo12x", task="detect")  # Load a model

exported_model_path = export_to_tensorrt(model, kwargs={
    "half": True,
})

for validation_dataset in validation_datasets:
    benchmark_result = benchmark(
        model=YOLO(exported_model_path, task="detect"),
        kwargs={
            "half": True,
            "data": validation_dataset,
        })
    
    results_data = {
        "mAP50-95": benchmark_result.results_dict.get('metrics/mAP50-95(B)', None),
        "mAP50": benchmark_result.results_dict.get('metrics/mAP50(B)', None),
        "latency_ms": benchmark_result.speed.get('inference', None),
        "fps": None,
    }
    if results_data["latency_ms"] is not None:
        results_data["fps"] = 1000.0 / results_data["latency_ms"]
    
    benchmark_results.append({
        "model": "yolo12x",
        "precision": "FP16",
        "dataset": validation_dataset,
        "benchmark_result": results_data,
    })
    
with open("benchmark_results_half.json", "w") as f:
    json.dump(benchmark_results, f, indent=4)

Ultralytics 8.3.109 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2070, 7789MiB)
YOLOv12x summary (fused): 283 layers, 59,135,744 parameters, 0 gradients, 199.0 GFLOPs

PyTorch: starting from 'yolo12x.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (113.8 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.50...
ONNX: export success ✅ 6.6s, saved as 'yolo12x.onnx' (226.0 MB)

TensorRT: starting export with TensorRT 10.9.0.34...
[05/01/2025-15:57:10] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU -928, GPU +188, now: CPU 3400, GPU 1044 (MiB)
[05/01/2025-15:57:11] [TRT] [I] ----------------------------------------------------------------
[05/01/2025-15:57:11] [TRT] [I] Input filename:   yolo12x.onnx
[05/01/2025-15:57:11] [TRT] [I] ONNX IR version:  0.0.9
[05/01/2025-15:57:11] [TRT] [I] Opset version:    19
[05/01/2025-15:57:11] [TRT] [I] Producer name:    pytorch
[05/01/2025-15:57:11] [TRT] [I] 

val: Scanning /home/omni/Programming/QRID/datasets/coco/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5000/5000 [01:48<00:00, 46.27it/s]


                   all       5000      36335      0.748      0.661      0.719      0.551
Speed: 0.3ms preprocess, 18.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Saving /home/omni/Programming/QRID/QRID/runs/detect/val73/predictions.json...

Evaluating pycocotools mAP using /home/omni/Programming/QRID/QRID/runs/detect/val73/predictions.json and /home/omni/Programming/QRID/datasets/coco/annotations/instances_val2017.json...
loading annotations into memory...
Done (t=0.37s)
creating index...
index created!
Loading and preparing results...
DONE (t=1.74s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=30.62s).
Accumulating evaluation results...
DONE (t=4.62s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.548
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.722
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.594
 Average Precision  (AP) @[ 

val: Scanning /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_low/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5000/5000 [01:42<00:00, 48.73it/s]


                   all       5000      36335      0.738      0.616      0.683      0.518
Speed: 0.3ms preprocess, 17.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Saving /home/omni/Programming/QRID/QRID/runs/detect/val74/predictions.json...

Evaluating pycocotools mAP using /home/omni/Programming/QRID/QRID/runs/detect/val74/predictions.json and /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_low/annotations/instances_val2017.json...
pycocotools unable to run: /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_low/annotations/instances_val2017.json file not found
Results saved to /home/omni/Programming/QRID/QRID/runs/detect/val74
Benchmark completed.
Benchmark results: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47

...

                   all       5000      36335       0.75      0.657      0.715      0.547
Speed: 0.3ms preprocess, 18.0ms inference, 0.0ms loss, 1.0ms postprocess per image
Saving /home/omni/Programming/QRID/QRID/runs/detect/val77/predictions.json...

Evaluating pycocotools mAP using /home/omni/Programming/QRID/QRID/runs/detect/val77/predictions.json and /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_jpeg_heavy/annotations/instances_val2017.json...
pycocotools unable to run: /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_jpeg_heavy/annotations/instances_val2017.json file not found
Results saved to /home/omni/Programming/QRID/QRID/runs/detect/val77
Benchmark completed.
Benchmark results: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47

val: Scanning /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_low/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5000/5000 [01:50<00:00, 45.18it/s]


                   all       5000      36335      0.556      0.311      0.335      0.231
Speed: 0.3ms preprocess, 18.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Saving /home/omni/Programming/QRID/QRID/runs/detect/val78/predictions.json...

Evaluating pycocotools mAP using /home/omni/Programming/QRID/QRID/runs/detect/val78/predictions.json and /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_low/annotations/instances_val2017.json...
pycocotools unable to run: /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_low/annotations/instances_val2017.json file not found
Results saved to /home/omni/Programming/QRID/QRID/runs/detect/val78
Benchmark completed.
Benchmark results: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 

val: Scanning /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_medium/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5000/5000 [01:51<00:00, 44.76it/s]


                   all       5000      36335      0.535      0.305      0.326      0.224
Speed: 0.3ms preprocess, 18.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Saving /home/omni/Programming/QRID/QRID/runs/detect/val79/predictions.json...

Evaluating pycocotools mAP using /home/omni/Programming/QRID/QRID/runs/detect/val79/predictions.json and /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_medium/annotations/instances_val2017.json...
pycocotools unable to run: /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_medium/annotations/instances_val2017.json file not found
Results saved to /home/omni/Programming/QRID/QRID/runs/detect/val79
Benchmark completed.
Benchmark results: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46

In [2]:
import json
from ultralytics import YOLO
from functions.benchmark import benchmark
from functions.export_to_tensorrt import export_to_tensorrt

calibration_sets = [
    "/home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_clean/data.yaml",
    "/home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_mixed/data.yaml",
]

validation_datasets = [
    # "coco.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_low/data_blurry_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_blurry_medium/data_blurry_medium.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_contrast_low/data_contrast_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_jpeg_heavy/data_jpeg_heavy.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_low/data_noisy_low.yaml",
    # "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_noisy_medium/data_noisy_medium.yaml"
    "/home/omni/Programming/QRID/QRID/validation_datasets/coco_val_mixed_degrad_50pct/data_coco_val_mixed_degrad_50pct.yaml",
]

benchmark_results = []

model = YOLO("yolo12x", task="detect")  # Load a model

for calibration_set in calibration_sets:
    exported_tensorrt_model = export_to_tensorrt(model, kwargs={
        "int8": True,
        "data": calibration_set,
    })

    for validation_dataset in validation_datasets:
        benchmark_result = benchmark(
            model=YOLO(exported_model_path, task="detect"),
            kwargs={
                "data": validation_dataset,
        })
        results_data = {
            "mAP50-95": benchmark_result.results_dict.get('metrics/mAP50-95(B)', None),
            "mAP50": benchmark_result.results_dict.get('metrics/mAP50(B)', None),
            "latency_ms": benchmark_result.speed.get('inference', None),
            "fps": None,
        }
        if results_data["latency_ms"] is not None:
            results_data["fps"] = 1000.0 / results_data["latency_ms"]
        
        benchmark_results.append({
            "model": "yolo12x",
            "calibration_set": calibration_set,
            "dataset": validation_dataset,
            "benchmark_result": results_data,
        })
        
with open("benchmark_results_static.json", "w") as f:
    json.dump(benchmark_results, f, indent=4)

Ultralytics 8.3.109 🚀 Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2070, 7789MiB)
YOLOv12x summary (fused): 283 layers, 59,135,744 parameters, 0 gradients, 199.0 GFLOPs

PyTorch: starting from 'yolo12x.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (113.8 MB)

ONNX: starting export with onnx 1.17.0 opset 19...
ONNX: slimming with onnxslim 0.1.50...
ONNX: export success ✅ 25.2s, saved as 'yolo12x.onnx' (226.0 MB)

TensorRT: starting export with TensorRT 10.9.0.34...
TensorRT: collecting INT8 calibration images from 'data=/home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_clean/data.yaml'


Scanning /home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_clean/labels/val2017.cache... 0 images, 1000 backgrounds, 0 corrupt: 100%|██████████| 1000/1000 [00:00<?, ?it/s]

WARNING ⚠️ No labels found in /home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_clean/labels/val2017.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.


[05/01/2025-17:19:44] [TRT] [I] The logger passed into createInferBuilder differs from one already provided for an existing builder, runtime, or refitter. Uses of the global logger, returned by nvinfer1::getLogger(), will return the existing value.
[05/01/2025-17:19:45] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU -784, GPU +188, now: CPU 3933, GPU 1736 (MiB)
[05/01/2025-17:19:45] [TRT] [I] ----------------------------------------------------------------
[05/01/2025-17:19:45] [TRT] [I] Input filename:   yolo12x.onnx
[05/01/2025-17:19:45] [TRT] [I] ONNX IR version:  0.0.9
[05/01/2025-17:19:45] [TRT] [I] Opset version:    19
[05/01/2025-17:19:45] [TRT] [I] Producer name:    pytorch
[05/01/2025-17:19:45] [TRT] [I] Producer version: 2.6.0
[05/01/2025-17:19:45] [TRT] [I] Domain:           
[05/01/2025-17:19:45] [TRT] [I] Model version:    0
[05/01/2025-17:19:45] [TRT] [I] Doc string:       
[05/01/2025-17:19:45] [TRT] [I] ------------------------------------------------------

val: Scanning /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_mixed_degrad_50pct/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5000/5000 [01:43<00:00, 48.10it/s]


...

Scanning /home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_mixed/labels/val2017.cache... 0 images, 1000 backgrounds, 0 corrupt: 100%|██████████| 1000/1000 [00:00<?, ?it/s]

WARNING ⚠️ No labels found in /home/omni/Programming/QRID/QRID/calibration_sets/coco_calib_mixed/labels/val2017.cache, training may not work correctly. See https://docs.ultralytics.com/datasets for dataset formatting guidance.
[05/01/2025-17:37:24] [TRT] [I] [MemUsageChange] Init builder kernel library: CPU -1138, GPU +172, now: CPU 6616, GPU 1286 (MiB)


[05/01/2025-17:37:24] [TRT] [I] ----------------------------------------------------------------
[05/01/2025-17:37:24] [TRT] [I] Input filename:   yolo12x.onnx
[05/01/2025-17:37:24] [TRT] [I] ONNX IR version:  0.0.9
[05/01/2025-17:37:24] [TRT] [I] Opset version:    19
[05/01/2025-17:37:24] [TRT] [I] Producer name:    pytorch
[05/01/2025-17:37:24] [TRT] [I] Producer version: 2.6.0
[05/01/2025-17:37:24] [TRT] [I] Domain:           
[05/01/2025-17:37:24] [TRT] [I] Model version:    0
[05/01/2025-17:37:24] [TRT] [I] Doc string:       
[05/01/2025-17:37:24] [TRT] [I] ----------------------------------------------------------------
TensorRT: input "images" with shape(-1, 3, -1, -1) DataType.FLOAT
TensorRT: output "output0" with shape(-1, 84, -1) DataType.FLOAT
TensorRT: WARNING ⚠️ 'dynamic=True' model requires max batch size, i.e. 'batch=16'
TensorRT: building INT8 engine as yolo12x.engine
[05/01/2025-17:37:24] [TRT] [I] BuilderFlag::kTF32 is set but hardware does not support TF32. Disabling

val: Scanning /home/omni/Programming/QRID/QRID/validation_datasets/coco_val_mixed_degrad_50pct/labels/val2017.cache... 4952 images, 48 backgrounds, 0 corrupt: 100%|██████████| 5000/5000 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5000/5000 [01:44<00:00, 47.87it/s]


                   all       5000      36335      0.695      0.537      0.602      0.451
Speed: 0.2ms preprocess, 18.0ms inference, 0.0ms loss, 0.9ms postprocess per image
Benchmark completed.
Benchmark results: ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77,
       78, 79])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7926c67a2690>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008